# Long-Term Memory — Semantic — LangGraph Agent Tutorial

Semantic memory stores **timeless facts** about a user ("Alice is a data engineer," "prefers
Python over Scala") that should be available across *every* future thread. In LangGraph this is
the canonical use case for the cross-thread **`Store`** — no checkpointer involved, since
checkpointers are thread-scoped by design.

This notebook builds a real tool-calling LangGraph agent: the LLM itself decides when to save a
fact and when to recall one, via tools backed by `SqliteStore`, namespaced by `user_id`.

In [3]:
# ============ IMPORTS ============
import os
import sys
import sqlite3
import json as jsonlib

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.store.sqlite import SqliteStore
from langgraph.config import get_store

sys.path.append(os.path.abspath("../../.."))
from helpers import get_llm

from dotenv import load_dotenv
load_dotenv()

print("Imports OK")

Imports OK


In [4]:
# ============ LLM + STORE INITIALIZATION ============
llm = get_llm()

DB_PATH = "semantic_memory.db"
conn = sqlite3.connect(DB_PATH, check_same_thread=False, isolation_level=None)
semantic_store = SqliteStore(conn)   # no TTL -- semantic memory is meant to be durable
semantic_store.setup()
print("semantic store ready")

LLM initialized: system.ai.gemma-3-12b (via databricks_gateway)
semantic store ready


## 1. Memory Tools Backed by the Store

Each tool calls `get_store()` to grab the store, instead of the store being passed in as a
function argument. Think of it as a "current store" that LangGraph makes available while the
graph is running — a tool just asks for it when it needs it, no wiring required.

The `user_id` never comes from the model. The agent node reads it from `config` (trusted,
server-side) and stashes it in a small module-level variable that the tools read from. So the
model can't make up a different `user_id` in a tool call and read someone else's memory.

Each fact is saved under a short, fixed `key`, like `"name"` or `"preferred_language"`. If the
same fact gets saved again later, it overwrites that same entry instead of adding a new one.
Skip this and facts start piling up as duplicates — see the Gotchas section below.

In [5]:
# ============ MEMORY TOOLS ============
_CURRENT_USER: dict = {"id": None}

@tool
def save_user_fact(key: str, fact: str) -> str:
    """Persist a durable fact about the user under a short stable key
    (e.g. key='name', key='preferred_language') so re-saving updates it in place."""
    store = get_store()
    store.put(("semantic", _CURRENT_USER["id"]), key, {"fact": fact})
    return f"Saved fact [{key}]: {fact}"

@tool
def recall_user_facts() -> str:
    """Return every stored fact about the current user."""
    store = get_store()
    items = store.search(("semantic", _CURRENT_USER["id"]))
    facts = {item.key: item.value["fact"] for item in items}
    return jsonlib.dumps(facts) if facts else "No facts stored yet."

semantic_tools = [save_user_fact, recall_user_facts]

## 2. The Agent — Preload + Tool Access

Rather than making the model call `recall_user_facts` on every turn, the agent node preloads
everything already known about the user straight from the store into the system prompt. Tools
stay available for the model to use when it learns something *new*.

In [6]:
# ============ MEMORY-AWARE AGENT GRAPH ============
llm_with_tools = llm.bind_tools(semantic_tools)

SYSTEM_TEMPLATE = """You are a helpful assistant with persistent semantic memory.

Known facts about this user:
{facts}

When the user reveals a durable fact about themselves (name, role, preference), call
`save_user_fact` with a short stable key so it updates cleanly next time.
"""

def agent_node(state: MessagesState, config: RunnableConfig) -> dict:
    user_id = config["configurable"]["user_id"]
    _CURRENT_USER["id"] = user_id

    store = get_store()
    items = store.search(("semantic", user_id))
    facts_block = "\n".join(f"- {it.value['fact']}" for it in items) or "(none)"

    system = SystemMessage(SYSTEM_TEMPLATE.format(facts=facts_block))
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(semantic_tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, ["tools", END])
builder.add_edge("tools", "agent")

semantic_agent = builder.compile(store=semantic_store)
print("Semantic-memory agent compiled.")

Semantic-memory agent compiled.


## 3. Conversation 1 — The Agent Learns About Alice

In [7]:
# ============ CONVERSATION 1: TEACH THE AGENT ============
USER_ID = "user-alice"
cfg1 = {"configurable": {"user_id": USER_ID, "thread_id": "alice-thread-1"}}

out = semantic_agent.invoke(
    {"messages": [HumanMessage("Hi, I'm Alice. I work as a data engineer and prefer Python over Scala.")]},
    cfg1,
)
print("AGENT:", out["messages"][-1].content)

AGENT: Okay, I've saved that you are Alice, a data engineer who prefers Python. Do you have any other preferences or details you'd like to share?


## 4. Conversation 2 — Brand-New Thread, Same User

Short-term memory (the checkpointer) isn't even wired up in this graph — there's nothing to
carry over between threads except what the **store** persists. This is the cleanest possible
proof that semantic memory survives independently of any thread.

In [8]:
# ============ CONVERSATION 2: NEW THREAD, MEMORY PERSISTS ============
cfg2 = {"configurable": {"user_id": USER_ID, "thread_id": "alice-thread-2"}}
out2 = semantic_agent.invoke(
    {"messages": [HumanMessage("What's my name and which language do I prefer?")]},
    cfg2,
)
print("AGENT:", out2["messages"][-1].content)

AGENT: Your name is Alice, and you prefer Python.


## 5. User Isolation

A different `user_id` — even mid-way through the exact same process — sees nothing.

In [9]:
# ============ ISOLATION CHECK ============
cfg_bob = {"configurable": {"user_id": "user-bob", "thread_id": "bob-thread-1"}}
out_bob = semantic_agent.invoke(
    {"messages": [HumanMessage("What do you know about me?")]},
    cfg_bob,
)
print("AGENT (user-bob):", out_bob["messages"][-1].content)

AGENT (user-bob): I don't know anything about you yet. I have no stored facts.


## 6. Update-in-Place, Inspected Directly

Alice restates her language preference. Because the tool always writes to the same `key`
(`"preferred_language"`), the store holds one row for it, not two — inspect the raw table to
prove it.

In [10]:
# ============ UPDATE-IN-PLACE DEMO ============
out3 = semantic_agent.invoke(
    {"messages": [HumanMessage("Actually, just to confirm again: I prefer Python over Scala.")]},
    cfg1,
)
print("AGENT:", out3["messages"][-1].content)

rows = conn.execute(
    "SELECT key, value FROM store WHERE prefix = ?", (f"semantic.{USER_ID}",)
).fetchall()
print(f"\n{len(rows)} row(s) for {USER_ID} -- restating the same fact did NOT create a duplicate:")
for key, value in rows:
    print(" -", key, "=", value)

AGENT: Okay, I've saved that you prefer Python. I also know you're a data engineer and your name is Alice. Is there anything else you'd like me to remember?

3 row(s) for user-alice -- restating the same fact did NOT create a duplicate:
 - name = b'{"fact":"Alice"}'
 - role = b'{"fact":"data engineer"}'
 - preferred_language = b'{"fact":"Python"}'


## Gotchas

- **Namespace tuples are a design decision, not an implementation detail.** `("semantic",
  user_id)` isolates users; get this wrong (e.g. a global namespace, or `thread_id` instead of
  `user_id`) and facts leak across users or vanish per-thread instead of persisting.
- **`get_store()` only works inside a graph run.** It reads from the run's contextvar-backed
  store — calling it outside of a node/tool invocation raises, and calling it from a tool that
  was invoked outside `ToolNode`'s normal flow won't have a store bound either.
- **Update-in-place depends entirely on using a stable `key`.** `store.put(namespace, key, ...)`
  overwrites when the key repeats; if the model (or your tool signature) generates a fresh
  `key`/random id per save instead of reusing `"name"`/`"preferred_language"`/etc., you silently
  get the same keyless-duplicate problem described in `00_Memory_Layers_Guide.md` — nothing in
  the Store API prevents it, the discipline has to come from how the tool is written.
- **Preloading everything doesn't scale.** `store.search()` with no `query`/`filter` returns the
  whole namespace — fine for a handful of facts, not for hundreds. At that point use `query=`
  (if the store has an embedding `index` configured) instead of blind preload.
- **`user_id` must be injected from trusted config, never read from the model's own tool-call
  arguments** — this notebook's tools deliberately take no `user_id` parameter at all, reading it
  only from the closed-over `_CURRENT_USER`, so the model has no lever to target another user's
  memory.

## Key Takeaways

- Semantic memory in LangGraph = a `Store` (here, `SqliteStore`, no TTL) namespaced by
  `user_id`, read/written through tools the model calls itself.
- `get_store()` inside a `@tool` is the idiomatic way to reach the store — no manual plumbing of
  a store object through function signatures.
- A stable `key` per fact is what makes "update in place" work; the Store API doesn't enforce it
  for you.
